# Convert GTFS `.txt` files to `.xlsx`

This notebook reads GTFS `.txt` files from:

`C:\Users\Sarad\Escritorio\Mates\Cursos\Curs 2025-2026 (3r + 4t)\TFG\TFG\.src\gtfs\data`

and exports them to Excel in a clean and reproducible way.

It includes:
1. One `.xlsx` file per `.txt` table.
2. One optional combined `.xlsx` file with one sheet per table.

In [ ]:
from pathlib import Path
import pandas as pd

# Source folder with GTFS text files
DATA_DIR = Path(r"C:\Users\Sarad\Escritorio\Mates\Cursos\Curs 2025-2026 (3r + 4t)\TFG\TFG\.src\gtfs\data")

# Output folder for generated Excel files
OUTPUT_DIR = DATA_DIR / "excel_exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

txt_files = sorted(DATA_DIR.glob("*.txt"))

if not txt_files:
    raise FileNotFoundError(f"No .txt files were found in {DATA_DIR}")

print(f"Input folder: {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"TXT files found: {len(txt_files)}")
for file_path in txt_files:
    print(f" - {file_path.name}")

In [ ]:
def read_txt_table(file_path: Path) -> pd.DataFrame:
    """Read a GTFS TXT file using a robust fallback for encoding."""
    try:
        return pd.read_csv(file_path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, low_memory=False, encoding="latin1")


conversion_log = []

for txt_file in txt_files:
    df = read_txt_table(txt_file)
    output_file = OUTPUT_DIR / f"{txt_file.stem}.xlsx"
    df.to_excel(output_file, index=False)

    conversion_log.append((txt_file.name, output_file.name, len(df), len(df.columns)))
    print(f"Converted {txt_file.name} -> {output_file.name}")

summary_df = pd.DataFrame(
    conversion_log,
    columns=["input_txt", "output_xlsx", "rows", "columns"],
)
summary_df

In [ ]:
def excel_sheet_name(base_name: str, used_names: set[str]) -> str:
    """Create a valid, unique Excel sheet name (max 31 chars)."""
    cleaned = base_name.replace("/", "_").replace("\\", "_").replace("*", "_")
    cleaned = cleaned.replace("?", "_").replace("[", "_").replace("]", "_").replace(":", "_")
    cleaned = cleaned[:31] if cleaned else "Sheet"

    candidate = cleaned
    counter = 1
    while candidate in used_names:
        suffix = f"_{counter}"
        candidate = f"{cleaned[:31-len(suffix)]}{suffix}"
        counter += 1

    used_names.add(candidate)
    return candidate


combined_file = OUTPUT_DIR / "gtfs_all_tables.xlsx"
used_sheet_names: set[str] = set()

with pd.ExcelWriter(combined_file, engine="openpyxl") as writer:
    for txt_file in txt_files:
        df = read_txt_table(txt_file)
        sheet = excel_sheet_name(txt_file.stem, used_sheet_names)
        df.to_excel(writer, sheet_name=sheet, index=False)
        print(f"Added sheet '{sheet}' from {txt_file.name}")

print(f"Combined workbook created: {combined_file}")

## Notes

- If you only need separate files, run the first two code cells.
- If you also want one file with all tables as sheets, run the third code cell too.
- Output location:
  - `...\.src\gtfs\data\excel_exports`